# Module 1 - From raw files to a Google OKF knowledge base

```
data/raw_syone_corp/        <- scraped web pages (Markdown)         \
data/raw_hardware_manuals/  <- 8 PDF manuals with tables + figures   /  Stage 1: acquisition
        |  Docling  (layout model + TableFormer tables + OCR)
data/docling_output/        <- Stage 2: structured JSON (markdown, tables, figures)
        |  OKF compiler (frontmatter, entities, related links, INDEX.md)
data/okf_store/             <- Stage 3: the knowledge base both retrieval tracks use
```
Run the cells in order and **read the outputs**: the point of this module is to see what each stage keeps and what it loses.

In [ ]:
import os, sys, json, asyncio, time, logging
from pathlib import Path
ROOT = Path("..").resolve()
sys.path.insert(0, str(ROOT))
os.chdir(ROOT)                                   # every path below is relative to the project root
os.environ.setdefault("HF_HOME", str(ROOT / "models" / "huggingface"))
logging.getLogger("neo4j").setLevel(logging.ERROR)
MODEL = os.getenv("FORMATION_MODEL", "formation-fast")   # use "formation-local" if you run LM Studio / Ollama

from backend.ingestion.docling_parser import DoclingDocumentParser
print("project root:", ROOT)

## 1. What did the acquisition scripts produce?

In [ ]:
manifest = json.loads((ROOT / "data/raw_syone_corp/_manifest.json").read_text())
print(f"Syone website: scraped {manifest['scraped_at'][:10]} from {manifest['source']}")
print(f"  {manifest['pages_ok']} pages kept, {manifest.get('pages_skipped', 0)} skipped (index/duplicates), "
      f"{manifest['pages_failed']} failed, {manifest['images']} informative images")
import collections
print("  pages per section:", dict(collections.Counter(p["section"] for p in manifest["pages"] if p["status"] == "ok").most_common(8)))
print()
for pdf in sorted((ROOT / "data/raw_hardware_manuals").glob("*.pdf")):
    print(f"  {pdf.name:45s} {pdf.stat().st_size/1024:6.0f} KB")

## 2. Parse a scraped page (no OCR needed) and a PDF (needs Docling)

In [ ]:
parser = DoclingDocumentParser(ROOT / "data/extracted_images", ocr_engine="docling_standard")

page = parser.parse_document(ROOT / "data/raw_syone_corp/partners-zabbix-partner.md", corpus="syone_corp")
print("TITLE  :", page["title"]); print("SOURCE :", page["source_url"]); print("SECTION:", page["section"]); print("ENGINE :", page["ocr_engine_used"])
print(page["markdown_content"][:700])

In [ ]:
t0 = time.time()
doc = parser.parse_document(ROOT / "data/raw_hardware_manuals/aor3_06_fault_code_reference.pdf", corpus="hardware_manuals")
print(f"{doc['ocr_engine_used']}  |  {time.time()-t0:.0f}s  |  {len(doc['markdown_content']):,} chars  |  {len(doc['tables'])} tables  |  {len(doc['images'])} figures")
print()
row = next(l for l in doc["markdown_content"].splitlines() if "0x4A7B_VRM_OVERVOLT" in l and l.startswith("|"))
print("A fault-code table row, extracted from the PDF as a Markdown table row:\n", row)

### Check what the parser kept: tables and figures
Two things commonly get lost in PDF extraction: **table structure** (rows become loose text) and **figures** (images become `<!-- image -->`).
Here every figure is bound to its file through the caption line, and its *description* is stored next to it, so the words drawn **inside** the picture become searchable text.

In [ ]:
import pandas as pd
print("First extracted table as a DataFrame:")
display(pd.DataFrame(doc["tables"][0]).head(4))
for im in doc["images"]:
    print(f"\n{im['figure_id']}: {im['caption']}\n   path: {im['path']}\n   description: {im['description'][:200]}...")
from IPython.display import Image, display
display(Image(filename=str(ROOT / "data/extracted_images/hardware_manuals" / doc["images"][0]["file"]), width=620))

## 3. Why Docling standard, and not a vision model?

A newer option is a **vision-language model** that reads a page *image* and writes Markdown. We measured one, GLM-OCR (0.9B), through Docling's VLM pipeline on the 8 Aurora manuals (Apple M4 Max), then removed it from this repository:

| | Docling standard (used here) | GLM-OCR, served on the GPU by LM Studio |
|---|---|---|
| What it is | Layout model + TableFormer (tables) + RapidOCR | 0.9B vision-language model: page image in, Markdown out |
| Time, 8 manuals | **31 s** | **569 s** (18 times slower; about 6 min/page on the CPU) |
| Table rows recovered | 427 | 368 |
| Fault-code manual | 6 tables | 4: the continuation of the fault table on the next page is lost |
| Document titles | the real titles | the file name |

**Newer is not better by default: measure on your own documents.** Vision models earn their place on scans, formulas and hard layouts; for born-digital PDFs with tables, the classic pipeline wins here.

## 4. Run the full pipeline: Docling -> JSON -> OKF

In [ ]:
from backend.ingestion.run_docling import run_docling_parsing
from backend.ingestion.run_okf import run_okf_compilation
logging.getLogger().setLevel(logging.WARNING)
n = run_docling_parsing("docling_standard", "all")
m = run_okf_compilation("docling_standard", "all")
print(f"parsed {n} documents, compiled {m} OKF files")

In [ ]:
okf = (ROOT / "data/okf_store/hardware_manuals/aor3_04_network_switch_manual.md").read_text()
print(okf.split("---")[1][:1800])           # the YAML frontmatter: type, entities, related links, figures
print("..."); print((ROOT / "data/okf_store/syone_corp/INDEX.md").read_text()[:900])   # the agent's entry point

In [ ]:
import subprocess
r = subprocess.run([sys.executable, "scripts/validate_dataset.py"], capture_output=True, text=True)
print(r.stdout)    # every fault code intact + every golden question answerable from this store

### Exercises
1. Open `data/okf_store/hardware_manuals/aor3_06_fault_code_reference.md`. Find a code whose cell text was wrapped across lines in the PDF. Did the parser keep it in one piece? (`validate_dataset.py` checks all 30.)
2. Add a ninth PDF of your own to `data/raw_hardware_manuals/`, re-run section 4 and look at its `entities:` list.
3. Change `okf_compiler._related()` so that same-section documents score higher. How do the `related:` lists change?